# Tugas Besar Data Sains — Analisis Struktur Data, EDA, dan Pre-processing
### Dataset: UK Road Safety Statistics (2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Garnie104/Data_Sains_2026/blob/main/Tugas/Tugas%20Besar/Hasil%20jadi/Tugas_Besar_Data_Sains.ipynb)

**Ruang Lingkup Tugas (Pengganti UTS):**
1. **Struktur Data**: Inspeksi skema, tipe data, ukuran dataset, serta relasi antar tabel (`collision`, `vehicle`, `casualty`).
2. **Exploratory Data Analysis (EDA)**: Statistik deskriptif, visualisasi distribusi, **deteksi outlier**, dan **analisis korelasi** (beserta penjelasan fitur yang dimasukkan).
3. **Data Pre-processing** *(Mengacu Materi Pertemuan 1–7 / Gabungan Slide Pak Surya & LMS)*:
   - *Data Selection & Merging*
   - *Data Cleaning & Validation*
   - *Imputasi Missing Values* (sesuai materi `Week 6` & `Pertemuan 09`)
   - *Data Construction*: **One-Hot Encoding** (`Pertemuan 09` hal. 70–75)
4. **Catatan:** Dataset ini berbentuk **tabular** (bukan data citra/gambar), sehingga teknik seperti *Image Segmentation* tidak diterapkan. Tahap *Machine Learning / Modeling* belum dilakukan pada laporan ini (akan dilanjutkan menjelang UAS).


## 0. Persiapan Lingkungan & Pemuatan Dataset
Kode di bawah ini otomatis mendeteksi apakah notebook dijalankan di **Google Colab** atau secara **lokal**. Jika di Google Colab, dataset akan otomatis diunduh dari repositori GitHub.


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

# Pengaturan visualisasi agar rapi untuk laporan cetak (hardcopy)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300
sns.set_theme(style='whitegrid', palette='muted')

# Deteksi lokasi dataset (Lokal vs Google Colab)
local_candidates = [
    '../../../dataset kecelakan UK',
    '../../dataset kecelakan UK',
    'dataset kecelakan UK'
]
colab_repo_path = 'Data_Sains_2026/dataset kecelakan UK'

DATA_DIR = None
for p in local_candidates:
    if os.path.exists(p):
        DATA_DIR = p
        print(f"Mode Lokal aktif. Dataset ditemukan di: {os.path.abspath(DATA_DIR)}")
        break

if DATA_DIR is None:
    if os.path.exists(colab_repo_path):
        DATA_DIR = colab_repo_path
        print(f"Mode Google Colab aktif. Dataset ditemukan di: {DATA_DIR}")
    else:
        print("Mengkloning dataset dari GitHub ke Google Colab...")
        os.system('git clone --depth 1 https://github.com/Garnie104/Data_Sains_2026.git')
        DATA_DIR = colab_repo_path
        print(f"Selesai! Dataset siap di: {DATA_DIR}")

OUTPUT_DIR = 'output_grafik'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Folder output grafik siap di: {OUTPUT_DIR}/")


---
## 1. Struktur Data (*Data Understanding*)
Dataset Kecelakaan Lalu Lintas UK Tahun 2025 terdiri dari 3 tabel relasional utama dan 1 file panduan (*Data Guide*):
1. `dft-road-casualty-statistics-collision-2025.csv` — Data utama per kejadian kecelakaan.
2. `dft-road-casualty-statistics-vehicle-2025.csv` — Data detail kendaraan yang terlibat.
3. `dft-road-casualty-statistics-casualty-2025.csv` — Data detail korban jiwa/luka.
4. `dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2025.xlsx` — Kamus kode kategori resmi.


In [ ]:
df_collision = pd.read_csv(os.path.join(DATA_DIR, 'dft-road-casualty-statistics-collision-2025.csv'), low_memory=False)
df_vehicle   = pd.read_csv(os.path.join(DATA_DIR, 'dft-road-casualty-statistics-vehicle-2025.csv'), low_memory=False)
df_casualty  = pd.read_csv(os.path.join(DATA_DIR, 'dft-road-casualty-statistics-casualty-2025.csv'), low_memory=False)

summary_tables = pd.DataFrame({
    'Tabel': ['Collision (Kecelakaan)', 'Vehicle (Kendaraan)', 'Casualty (Korban)'],
    'Jumlah Baris': [df_collision.shape[0], df_vehicle.shape[0], df_casualty.shape[0]],
    'Jumlah Kolom': [df_collision.shape[1], df_vehicle.shape[1], df_casualty.shape[1]],
    'Primary / Foreign Key': ['collision_index', 'collision_index, vehicle_reference', 'collision_index, vehicle_reference, casualty_reference']
})
display(summary_tables)


In [ ]:
print("=== Struktur Tipe Data: Tabel Collision ===")
print(df_collision.dtypes.value_counts())
print("\n5 Baris Pertama Tabel Collision:")
display(df_collision.head())


In [ ]:
print("=== Struktur Tipe Data: Tabel Vehicle & Casualty ===")
print("Vehicle dtypes:", df_vehicle.dtypes.value_counts().to_dict())
print("Casualty dtypes:", df_casualty.dtypes.value_counts().to_dict())

# Cek irisan collision_index antar tabel
idx_col = set(df_collision['collision_index'])
idx_veh = set(df_vehicle['collision_index'])
idx_cas = set(df_casualty['collision_index'])
print(f"\nKecocokan key collision_index:")
print(f"- Collision vs Vehicle  : {len(idx_col & idx_veh):,} dari {len(idx_col):,} kejadian")
print(f"- Collision vs Casualty : {len(idx_col & idx_cas):,} dari {len(idx_col):,} kejadian")


---
## 2. Exploratory Data Analysis (EDA)
### 2.1 Statistik Deskriptif Fitur Utama
Pada dataset resmi UK Department for Transport (DfT), nilai **`-1`** merepresentasikan *Data missing or out of range*. Pada tahap EDA awal ini kita lihat statistik mentahnya terlebih dahulu sebelum dibersihkan di tahap Pre-processing.


In [ ]:
num_cols_eda = ['collision_severity', 'number_of_vehicles', 'number_of_casualties', 'speed_limit', 'day_of_week', 'urban_or_rural_area']
display(df_collision[num_cols_eda].describe().T)


### 2.2 Visualisasi Distribusi Variabel Utama


In [ ]:
# Mapping label berdasarkan Data Guide UK 2025
severity_map = {1: 'Fatal (1)', 2: 'Serious (2)', 3: 'Slight (3)'}
day_map = {1: 'Minggu', 2: 'Senin', 3: 'Selasa', 4: 'Rabu', 5: 'Kamis', 6: 'Jumat', 7: 'Sabtu'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Distribusi Tingkat Keparahan Kecelakaan
sev_counts = df_collision['collision_severity'].map(severity_map).value_counts()
sns.barplot(x=sev_counts.index, y=sev_counts.values, ax=axes[0], palette=['#d62728', '#ff7f0e', '#1f77b4'])
axes[0].set_title('Distribusi Tingkat Keparahan Kecelakaan (2025)', fontweight='bold')
axes[0].set_xlabel('Tingkat Keparahan')
axes[0].set_ylabel('Jumlah Kejadian')
for i, v in enumerate(sev_counts.values):
    axes[0].text(i, v + 800, f"{v:,}\n({v/len(df_collision)*100:.1f}%)", ha='center', fontsize=9)

# Plot 2: Distribusi Kecelakaan per Hari
day_counts = df_collision['day_of_week'].map(day_map).value_counts().reindex(['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu'])
sns.barplot(x=day_counts.index, y=day_counts.values, ax=axes[1], color='#2b5c8f')
axes[1].set_title('Distribusi Jumlah Kecelakaan per Hari', fontweight='bold')
axes[1].set_xlabel('Hari')
axes[1].set_ylabel('Jumlah Kejadian')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '1_eda_distribusi_keparahan_hari.png'), bbox_inches='tight')
plt.show()


In [ ]:
# Plot 3: Tren Jam Kejadian Kecelakaan
df_collision['hour'] = pd.to_datetime(df_collision['time'], format='%H:%M', errors='coerce').dt.hour
hour_counts = df_collision['hour'].value_counts().sort_index()

plt.figure(figsize=(11, 4.5))
sns.lineplot(x=hour_counts.index, y=hour_counts.values, marker='o', linewidth=2.2, color='#c0392b')
plt.title('Tren Kejadian Kecelakaan Berdasarkan Jam (2025)', fontweight='bold')
plt.xlabel('Jam (00 - 23)')
plt.ylabel('Jumlah Kecelakaan')
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '2_eda_tren_jam.png'), bbox_inches='tight')
plt.show()


### 2.3 Deteksi Outlier (Metode IQR & Boxplot)
Deteksi outlier dilakukan pada fitur-fitur numerik kuantitatif utama dari ketiga tabel:
- Dari `collision`: `number_of_vehicles`, `number_of_casualties`, `speed_limit`
- Dari `vehicle`: `age_of_driver`, `engine_capacity_cc`, `age_of_vehicle` (nilai `-1` dikecualikan dari perhitungan outlier agar tidak bias sebagai angka negatif)
- Dari `casualty`: `age_of_casualty`


In [ ]:
def detect_outliers_iqr(series, name):
    s = series.dropna()
    s = s[s >= 0]  # Kecualikan kode missing -1
    if len(s) == 0:
        return {
            'Fitur': name, 'Q1': None, 'Q3': None, 'IQR': None,
            'Batas Bawah': None, 'Batas Atas': None,
            'Jumlah Outlier': 0, 'Persentase (%)': 0.0
        }
    Q1 = s.quantile(0.25)
    Q3 = s.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = s[(s < lower) | (s > upper)]
    return {
        'Fitur': name,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Batas Bawah': round(lower, 2),
        'Batas Atas': round(upper, 2),
        'Jumlah Outlier': len(outliers),
        'Persentase (%)': round(len(outliers) / len(s) * 100, 2)
    }

outlier_results = [
    detect_outliers_iqr(df_collision['number_of_vehicles'], 'number_of_vehicles'),
    detect_outliers_iqr(df_collision['number_of_casualties'], 'number_of_casualties'),
    detect_outliers_iqr(df_collision['speed_limit'], 'speed_limit'),
    detect_outliers_iqr(df_vehicle['age_of_driver'], 'age_of_driver'),
    detect_outliers_iqr(df_vehicle['engine_capacity_cc'], 'engine_capacity_cc'),
    detect_outliers_iqr(df_casualty['age_of_casualty'], 'age_of_casualty'),
]

df_outlier_summary = pd.DataFrame(outlier_results)
display(df_outlier_summary)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.boxplot(x=df_collision['number_of_vehicles'], ax=axes[0, 0], color='#5dade2')
axes[0, 0].set_title('Boxplot: Jumlah Kendaraan (number_of_vehicles)')

sns.boxplot(x=df_collision['number_of_casualties'], ax=axes[0, 1], color='#ec7063')
axes[0, 1].set_title('Boxplot: Jumlah Korban (number_of_casualties)')

sns.boxplot(x=df_vehicle.loc[df_vehicle['age_of_driver'] >= 0, 'age_of_driver'], ax=axes[1, 0], color='#58d68d')
axes[1, 0].set_title('Boxplot: Usia Pengemudi (age_of_driver)')

sns.boxplot(x=df_vehicle.loc[df_vehicle['engine_capacity_cc'] >= 0, 'engine_capacity_cc'], ax=axes[1, 1], color='#f5b041')
axes[1, 1].set_title('Boxplot: Kapasitas Mesin (engine_capacity_cc)')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '3_eda_boxplot_outlier.png'), bbox_inches='tight')
plt.show()


### 2.4 Analisis Korelasi & Daftar Fitur yang Masuk dalam Korelasi
Sesuai instruksi tugas, **tidak semua kolom** dimasukkan ke dalam perhitungan korelasi:
- **Fitur yang DIKELUARKAN:**
  - Kolom ID/referensi (`collision_index`, `collision_ref_no`) karena berupa pengenal unik.
  - Kolom kategorikal nominal murni tanpa urutan numerik bermakna (misalnya kode wilayah `local_authority_district`, `police_force`, `lsoa_of_accident_location`) karena angka kodenya bersifat arbitrer.
  - Kolom konstan (`collision_year` seluruhnya 2025).
- **Fitur yang DIMASUKKAN ke Korelasi (Numerik & Ordinal Bermakna):**
  1. `collision_severity` (1=Fatal, 2=Serious, 3=Slight — ordinal)
  2. `number_of_vehicles` (numerik diskrit)
  3. `number_of_casualties` (numerik diskrit)
  4. `speed_limit` (numerik rasio/ordinal)
  5. `urban_or_rural_area` (1=Urban, 2=Rural)
  6. `light_conditions` (kondisi pencahayaan)
  7. `weather_conditions` (kondisi cuaca)
  8. `road_surface_conditions` (kondisi permukaan jalan)
  9. `hour` (jam kejadian hasil ekstraksi waktu)
  10. `avg_driver_age` (rata-rata usia pengemudi per kejadian dari tabel `vehicle`)
  11. `avg_engine_cc` (rata-rata kapasitas mesin kendaraan per kejadian dari tabel `vehicle`)
  *(Catatan: `age_of_vehicle` tidak dimasukkan karena pada dataset 2025 seluruh nilainya adalah `-1`/kosong).*


In [ ]:
# Agregasi fitur numerik dari tabel vehicle per collision_index (mengecualikan nilai -1)
veh_valid = df_vehicle.copy()
veh_valid['age_of_driver'] = veh_valid['age_of_driver'].replace(-1, np.nan)
veh_valid['age_of_vehicle'] = veh_valid['age_of_vehicle'].replace(-1, np.nan)

veh_agg = veh_valid.groupby('collision_index').agg(
    avg_driver_age=('age_of_driver', 'mean'),
    avg_vehicle_age=('age_of_vehicle', 'mean'),
    avg_engine_cc=('engine_capacity_cc', lambda x: x[x > 0].mean())
).reset_index()

df_corr_base = df_collision.merge(veh_agg, on='collision_index', how='left')

corr_features = [
    'collision_severity', 'number_of_vehicles', 'number_of_casualties',
    'speed_limit', 'urban_or_rural_area', 'light_conditions',
    'weather_conditions', 'road_surface_conditions', 'hour',
    'avg_driver_age', 'avg_engine_cc'
]

# Ganti -1 dengan NaN sebelum menghitung korelasi agar kode missing tidak merusak korelasi
df_corr_clean = df_corr_base[corr_features].replace(-1, np.nan)
corr_matrix = df_corr_clean.corr(method='pearson')

plt.figure(figsize=(12, 9.5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', linewidths=0.5, square=True)
plt.title('Matriks Korelasi Fitur Numerik & Ordinal Kecelakaan UK (2025)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '4_eda_heatmap_korelasi.png'), bbox_inches='tight')
plt.show()


---
## 3. Data Pre-processing (Mengacu Slide 1–7 / Gabungan Pak Surya & LMS)
Tahap pre-processing dilakukan secara berurutan:
1. **Data Selection & Merging** (*Pertemuan 07 / Week 5*)
2. **Data Cleaning & Validation** (*Week 5*)
3. **Imputasi Missing Values** (*Week 6 & Pertemuan 09* — klarifikasi istilah `input` pada instruksi)
4. **Data Construction: One-Hot Encoding** (*Pertemuan 09* hal. 70–75)


### 3.1 Data Selection & Merging


In [ ]:
selected_cols = [
    'collision_index', 'date', 'day_of_week', 'hour',
    'collision_severity', 'number_of_vehicles', 'number_of_casualties',
    'road_type', 'speed_limit', 'light_conditions',
    'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area'
]

df_prep = df_collision[selected_cols].merge(veh_agg, on='collision_index', how='left')
print(f"Dimensi awal setelah selection & merge: {df_prep.shape}")


### 3.2 Data Cleaning & Validation
Langkah yang dilakukan:
- Menghapus data duplikat jika ada.
- Mengonversi kode khusus **`-1`** (serta nilai `9` / `99` yang berarti *unknown* pada beberapa kolom kategorikal tertentu jika diperlukan, di sini kita fokuskan pada `-1` sesuai standar *missing* DfT) menjadi `NaN`.
- Validasi tipe data tanggal (`date` -> `datetime`) dan validasi rentang nilai (`speed_limit` > 0, `avg_driver_age` 10–100 tahun, `urban_or_rural_area` dalam {1, 2}).


In [ ]:
# 1. Cek & hapus duplikat
dup_count = df_prep.duplicated(subset=['collision_index']).sum()
print(f"Jumlah duplikat collision_index: {dup_count}")
df_prep = df_prep.drop_duplicates(subset=['collision_index']).copy()

# 2. Konversi kode missing -1 menjadi NaN
df_prep = df_prep.replace(-1, np.nan)

# 3. Validasi rentang nilai
df_prep.loc[(df_prep['speed_limit'] <= 0) | (df_prep['speed_limit'] > 70), 'speed_limit'] = np.nan
df_prep.loc[~df_prep['urban_or_rural_area'].isin([1, 2]), 'urban_or_rural_area'] = np.nan
df_prep.loc[(df_prep['avg_driver_age'] < 10) | (df_prep['avg_driver_age'] > 105), 'avg_driver_age'] = np.nan
df_prep.loc[(df_prep['avg_vehicle_age'] < 0) | (df_prep['avg_vehicle_age'] > 60), 'avg_vehicle_age'] = np.nan

# 4. Konversi format tanggal
df_prep['date'] = pd.to_datetime(df_prep['date'], format='%d/%m/%Y', errors='coerce')
df_prep['month'] = df_prep['date'].dt.month

# 5. Drop kolom yang 100% kosong (misal avg_vehicle_age karena age_of_vehicle seluruhnya -1 pada data 2025)
all_null_cols = [c for c in df_prep.columns if df_prep[c].isnull().all()]
if all_null_cols:
    print(f"Menghapus kolom dengan 100% missing values: {all_null_cols}")
    df_prep = df_prep.drop(columns=all_null_cols)

# Tampilkan status missing values sebelum imputasi
missing_before = pd.DataFrame({
    'Missing Count': df_prep.isnull().sum(),
    'Missing (%)': (df_prep.isnull().sum() / len(df_prep) * 100).round(2)
})
display(missing_before[missing_before['Missing Count'] > 0])


### 3.3 Imputasi Missing Values (Klarifikasi istilah `input` pada instruksi)
Sesuai materi `Week 6 - SMC.pdf` dan `Pertemuan 09`:
- **Fitur Numerik** (`hour`, `speed_limit`, `avg_driver_age`, `avg_engine_cc`): Menggunakan **Imputasi Median** karena distribusi fitur-fitur ini memiliki *outlier* (skewed), sehingga median lebih tahan terhadap nilai ekstrem dibanding mean.
- **Fitur Kategorikal** (`road_type`, `light_conditions`, `weather_conditions`, `road_surface_conditions`, `urban_or_rural_area`): Menggunakan **Imputasi Modus (nilai terbanyak)**.


In [ ]:
numeric_impute_cols = ['hour', 'speed_limit', 'avg_driver_age', 'avg_engine_cc']
cat_impute_cols = ['road_type', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area']

for col in numeric_impute_cols:
    median_val = df_prep[col].median()
    df_prep[col] = df_prep[col].fillna(median_val)

for col in cat_impute_cols:
    mode_val = df_prep[col].mode()[0]
    df_prep[col] = df_prep[col].fillna(mode_val)

print("Total missing values setelah imputasi:", df_prep.isnull().sum().sum())


### 3.4 Data Construction — One-Hot Encoding (WAJIB)
Mengacu pada materi `Pertemuan 09` (hal. 70–75), fitur kategorikal nominal tidak boleh dibiarkan dalam bentuk kode angka mentah agar tidak disalahartikan sebagai urutan matematis. Kita terapkan **One-Hot Encoding** pada kolom kategorikal nominal berikut:
- `road_type`
- `weather_conditions`
- `light_conditions`
- `road_surface_conditions`
- `urban_or_rural_area`


In [ ]:
# Mapping label deskriptif agar nama kolom hasil One-Hot Encoding mudah dibaca di laporan
road_type_map = {1: 'Roundabout', 2: 'OneWayStreet', 3: 'DualCarriageway', 6: 'SingleCarriageway', 7: 'SlipRoad', 9: 'Unknown'}
weather_map = {1: 'Fine_NoHighWinds', 2: 'Raining_NoHighWinds', 3: 'Snowing_NoHighWinds', 4: 'Fine_HighWinds', 5: 'Raining_HighWinds', 6: 'Snowing_HighWinds', 7: 'FogOrMist', 8: 'Other', 9: 'Unknown'}
surface_map = {1: 'Dry', 2: 'WetOrDamp', 3: 'Snow', 4: 'FrostOrIce', 5: 'Flood', 9: 'Unknown'}
urban_map = {1: 'Urban', 2: 'Rural'}

df_encode = df_prep.copy()
df_encode['road_type'] = df_encode['road_type'].map(road_type_map).fillna('Other')
df_encode['weather_conditions'] = df_encode['weather_conditions'].map(weather_map).fillna('Other')
df_encode['road_surface_conditions'] = df_encode['road_surface_conditions'].map(surface_map).fillna('Other')
df_encode['urban_or_rural_area'] = df_encode['urban_or_rural_area'].map(urban_map).fillna('Other')
df_encode['light_conditions'] = 'Light_' + df_encode['light_conditions'].astype(int).astype(str)

ohe_cols = ['road_type', 'weather_conditions', 'light_conditions', 'road_surface_conditions', 'urban_or_rural_area']

print(f"Dimensi SEBELUM One-Hot Encoding: {df_encode.shape}")

# Menerapkan OneHotEncoder dari scikit-learn
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=int)
encoded_array = encoder.fit_transform(df_encode[ohe_cols])
encoded_col_names = encoder.get_feature_names_out(ohe_cols)

df_encoded_features = pd.DataFrame(encoded_array, columns=encoded_col_names, index=df_encode.index)
df_final = pd.concat([df_encode.drop(columns=ohe_cols), df_encoded_features], axis=1)

print(f"Dimensi SESUDAH One-Hot Encoding: {df_final.shape}")
print(f"Jumlah kolom baru hasil One-Hot Encoding: {len(encoded_col_names)}")
display(df_final[list(encoded_col_names[:10])].head())


---
## 4. Ringkasan Hasil Akhir Pre-processing (Siap untuk Tahap Modeling UAS)
Seluruh tahapan wajib untuk laporan Tugas Besar (Pengganti UTS) telah selesai dijalankan:
- Struktur data 3 tabel UK Road Safety 2025 berhasil dipetakan.
- EDA, Deteksi Outlier (IQR & Boxplot), dan Matriks Korelasi telah dibuat beserta alasan pemilihan fitur.
- Pre-processing lengkap (Selection, Merging, Cleaning/Validation kode `-1`, Imputasi Median/Modus, dan One-Hot Encoding) berhasil menghasilkan dataset bersih tanpa *missing values*.


In [ ]:
print("Shape dataset akhir:", df_final.shape)
print("Total missing values:", df_final.isnull().sum().sum())
print("Daftar grafik yang tersimpan di folder output_grafik/:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", f)
